# 📓 Caderno de Exercícios — Aula 03: Limpeza Numérica, Regras Matemáticas e Outliers
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **O Coração Financeiro da Base:** Agora vamos tratar `faturamento`, `margem_de_lucro` e `lucro`. Lembre-se da equação fundamental do negócio:
> $$\text{Lucro} = \text{Faturamento} \times \text{Margem de Lucro}$$

In [8]:
from difflib import get_close_matches
from pathlib import Path
import numpy as np
import pandas as pd

# === Replay Rápido das Aulas 01 e 02 ===
caminho_arquivo = Path(
    r"C:\Users\luisf\OneDrive\Documentos\Se2_Asimove\Campe_Supply_Corrompido.xlsx"
)
df = pd.read_excel(caminho_arquivo, sheet_name="Campe Supply")

df = df.drop(
    columns=["UNNAMED: 0", "coluna_inutil_0", "coluna_inutil_1"],
    errors="ignore",
)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("ã", "a")
)
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    "setor": ["Congelados", "Latícinio", "Bebidas", "Carnes"],
    "produto": [
        "Queijo",
        "Sorvete",
        "Frango",
        "Refrigerantes",
        "Cervejas",
        "Bovina",
        "Leite",
        "Lasanha",
        "Pizza",
        "Suína",
        "Sucos",
        "Coalhada",
    ],
    "vendedor": ["Vanessa", "Amaral", "Leo", "Daniele", "Jonas", "Evandro"],
    "regiao": ["Sudeste", "Nordeste", "Sul"],
    "estado": [
        "Rio de Janeiro",
        "Minas Gerais",
        "Bahia",
        "Rio Grande do Norte",
        "São Paulo",
        "Sergipe",
        "Rio Grande do Sul",
        "Paraná",
        "Santa Catarina",
    ],
    "cliente": ["Atacarejo", "Varejo", "Atacado"],
}


def corrigir_palavra(valor, lista_oficial):
  if pd.isna(valor):
    return np.nan
  match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
  return match[0] if match else np.nan


for col, lista_valida in categorias_oficiais.items():
  df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {
    "Queijo": "Latícinio",
    "Leite": "Latícinio",
    "Coalhada": "Latícinio",
    "Sorvete": "Congelados",
    "Lasanha": "Congelados",
    "Pizza": "Congelados",
    "Frango": "Carnes",
    "Bovina": "Carnes",
    "Suína": "Carnes",
    "Refrigerantes": "Bebidas",
    "Cervejas": "Bebidas",
    "Sucos": "Bebidas",
}
mapa_estado_regiao = {
    "Rio de Janeiro": "Sudeste",
    "Minas Gerais": "Sudeste",
    "São Paulo": "Sudeste",
    "Bahia": "Nordeste",
    "Rio Grande do Norte": "Nordeste",
    "Sergipe": "Nordeste",
    "Rio Grande do Sul": "Sul",
    "Paraná": "Sul",
    "Santa Catarina": "Sul",
}

df["setor"] = df["setor"].fillna(df["produto"].map(mapa_produto_setor))
df["regiao"] = df["regiao"].fillna(df["estado"].map(mapa_estado_regiao))

## 🎯 Missão 1: Investigando e Convertendo as Colunas Numéricas
As colunas `faturamento`, `margem_de_lucro` e `lucro` estão como `object` porque contêm erros de digitação com letras no meio dos números (ex: `"60b.72"`, `"0.a6"`).

1. Filtre ou exiba alguns desses valores corrompidos para ver como eles são.
2. Converta as 3 colunas para tipo numérico (`float`) usando `pd.to_numeric(..., errors='coerce')` (isso transformará os textos inválidos em `NaN` para podermos recalculá-los no próximo passo).
3. Rode um `df[['faturamento', 'margem_de_lucro', 'lucro']].describe()` e observe os valores mínimos e máximos!

In [10]:
# 1. Converta as 3 colunas financeiras para numérico e analise o .describe()
# Seu código aqui:
colunas_num = ['faturamento', 'margem_de_lucro', 'lucro']

for col in colunas_num:
    serie_convertida = pd.to_numeric(df[col], errors='coerce')
    valores_corrompidos = df[df[col].notna() & serie_convertida.isna()][col]
    print(f'Exemplos de textos corrompidos em {col.upper()} (Total: {len(valores_corrompidos)}):')
    print(valores_corrompidos.head(5).tolist())
    print('-'*50)

    # Convertendo para float (valores com letras viram NaN temporariamente)
for col in colunas_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Quantidade de nulos nas colunas numéricas logo após a conversão:')
print(df[colunas_num].isna().sum())

# Olhando as estatísticas descritivas antes de limpar outliers
df[colunas_num].describe()

Exemplos de textos corrompidos em FATURAMENTO (Total: 33):
['60b.72', '964.m699999999999', '9z5.29', '564u98', 'q78.55']
--------------------------------------------------
Exemplos de textos corrompidos em MARGEM_DE_LUCRO (Total: 29):
['han', '0.a6', 'q.11', '0.c4', 't.18']
--------------------------------------------------
Exemplos de textos corrompidos em LUCRO (Total: 32):
['98.275200000000b1', 't6.32639999999999', 'e46.1072', '72.016100000000h1', '1o2.4068']
--------------------------------------------------
Quantidade de nulos nas colunas numéricas logo após a conversão:
faturamento        115
margem_de_lucro    106
lucro              116
dtype: int64


,faturamento,margem_de_lucro,lucro
count,1540.000000,1549.000000,1.539000e+03
mean,896.140506,0.119464,3.542544e+03
std,401.069156,0.043099,9.783214e+04
min,520.360000,0.000000,2.669350e+01
25%,669.647500,0.080000,6.572055e+01
50%,798.740000,0.120000,9.550400e+01
75%,947.420000,0.160000,1.307138e+02
max,8581.600000,0.190000,3.260000e+06


## 🎯 Missão 2: Caçando e Corrigindo os 5 Outliers / Erros Matemáticos
Ao olhar o `.describe()`, você viu coisas absurdas como um **lucro máximo de R$ 3.260.000,00** e um **faturamento de R$ 8.581,60**!

1. Crie um filtro para encontrar todas as linhas onde a conta `faturamento * margem_de_lucro` difere de `lucro` em mais de R$ 0,01.
2. Você vai encontrar exatamente **5 linhas** com erro:
   * Em 2 linhas, o `lucro` está maior que o próprio `faturamento` (milhões). Corrija recalculando `lucro = faturamento * margem_de_lucro`.
   * Em 2 linhas, a `margem_de_lucro` está `0.00`, mas existe lucro positivo. Corrija recalculando `margem_de_lucro = lucro / faturamento`.
   * Em 1 linha, o `faturamento` foi multiplicado por 10 (`8581.60`), mas a margem (`0.06`) e o lucro (`51.4896`) estão certos. Corrija recalculando `faturamento = lucro / margem_de_lucro`.

In [14]:
# 2. Identifique as 5 linhas com erro matemático e corrija cada caso usando a fórmula do negócio
# Seu código aqui:
# Calculando a diferença absoluta entre (faturamento * margem) e lucro
diferenca_matematica = (df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs()

# Filtrando as linhas onde a diferença é maior que 1 centavo (0.01)
erros_matematicos = df[diferenca_matematica > 0.01]
erros_matematicos[['data_da_venda', 'produto', 'vendedor', 'faturamento', 'margem_de_lucro', 'lucro']]

# 1. Corrigindo lucros absurdos (onde lucro > faturamento)
mask_lucro_errado = df['lucro'] > df['faturamento']
df.loc[mask_lucro_errado, 'lucro'] = df.loc[mask_lucro_errado, 'faturamento'] * df.loc[mask_lucro_errado, 'margem_de_lucro']

# 2. Corrigindo margens zeradas onde há lucro positivo
mask_margem_zero = (df['margem_de_lucro'] == 0) & (df['lucro'] > 0)
df.loc[mask_margem_zero, 'margem_de_lucro'] = df.loc[mask_margem_zero, 'lucro'] / df.loc[mask_margem_zero, 'faturamento']

# 3. Corrigindo o faturamento discrepante (onde a conta ainda não bate)
mask_fat_errado = (df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01
df.loc[mask_fat_errado, 'faturamento'] = df.loc[mask_fat_errado, 'lucro'] / df.loc[mask_fat_errado, 'margem_de_lucro']

print('Linhas com erro matemático restantes:', ((df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01).sum())

Linhas com erro matemático restantes: 0


## 🎯 Missão 3: Recuperando Nulos Numéricos com Matemática
Agora que não há mais outliers mentirosos, você tem mais de 100 células `NaN` nas 3 colunas numéricas. Porém, na maioria dessas linhas, **2 das 3 colunas estão preenchidas**!

Use a equação $\text{Lucro} = \text{Faturamento} \times \text{Margem}$ para preencher os `NaN` de:
* `lucro` (usando `faturamento * margem_de_lucro`)
* `faturamento` (usando `lucro / margem_de_lucro`)
* `margem_de_lucro` (usando `lucro / faturamento`)

Depois, arredonde as 3 colunas para 2 casas decimais (`.round(2)`).
* ✅ **Checkpoint de Validação:** Os nulos de `faturamento` devem cair de **115 para 20**, os de `margem_de_lucro` de **106 para 16**, e os de `lucro` de **116 para 13**!

In [15]:
# 3. Recupere os valores nulos das 3 colunas financeiras e arredonde para 2 casas decimais
# Seu código aqui:
# Preenchendo nulos usando a relação matemática entre as 3 colunas
df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro'])
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro'])
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento'])

# Arredondando para 2 casas decimais
df['faturamento'] = df['faturamento'].round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].round(2)
df['lucro'] = df['lucro'].round(2)

print('Nulos numéricos APÓS a recuperação matemática:')
print(df[colunas_num].isna().sum())
df[colunas_num].describe()

Nulos numéricos APÓS a recuperação matemática:
faturamento        20
margem_de_lucro    16
lucro              13
dtype: int64


,faturamento,margem_de_lucro,lucro
count,1635.000000,1639.000000,1642.000000
mean,893.145480,0.119579,106.956784
std,348.922119,0.042827,59.135425
min,520.360000,0.050000,26.690000
25%,669.965000,0.080000,65.830000
50%,804.180000,0.120000,95.660000
75%,951.020000,0.160000,131.085000
max,2195.030000,0.190000,394.820000


## 🎯 Missão 4: A Segunda Onda de Duplicatas e Engenharia de Datas
Agora que corrigimos textos (`"Vacessa" -> "Vanessa"`) e números (`"60b.72" -> 604.72`), linhas que eram idênticas — mas estavam disfarçadas por erros de digitação — ficaram iguais!

1. Verifique quantas novas duplicatas surgiram com `df.duplicated().sum()` e remova-as com `df.drop_duplicates()`.
2. Crie uma coluna `mes` extraindo o mês de `data_da_venda` (`df['data_da_venda'].dt.month`).

* ✅ **Checkpoint Final da Limpeza:** Sua base limpa terá exatamente **1.541 linhas únicas**!

In [17]:
# 4. Remova as novas duplicatas reveladas, crie a coluna 'mes' e salve o Excel limpo se desejar
# Seu código aqui:
print('Novas duplicatas reveladas após a limpeza de textos e números:', df.duplicated().sum())

# Removendo as novas duplicatas
df = df.drop_duplicates().reset_index(drop=True)
print('Formato da base após 2ª desduplicação:', df.shape)


# Criando colunas temporais a partir de data_da_venda para usar nas análises da Aula 04 e 05
df['mes'] = df['data_da_venda'].dt.month
df['trimestre'] = df['data_da_venda'].dt.quarter

# Salvando a base limpa para usar nas próximas aulas
df.to_excel('Campe_Supply_Limpo.xlsx', index=False)
print('Arquivo Campe_Supply_Limpo.xlsx exportado com sucesso!')

Novas duplicatas reveladas após a limpeza de textos e números: 0
Formato da base após 2ª desduplicação: (1541, 10)
Arquivo Campe_Supply_Limpo.xlsx exportado com sucesso!
